In [ ]:
import os
import time
import yaml
import psutil
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

MAX_IMAGES = 50 

In [ ]:
MODEL_PATH = "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/runs/segment/stage1_head_warmup_7cls_extended/stage1_head_warmup_7cls_extended/weights/best.pt"
DATASET_YAML = "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/data/processed/yolo_seg_clean_2200_7cls/dataset.yaml"
SAHI_CONFIG_PATH = "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/configs/inference/sahi_production.yaml"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"✅ Using device: {device}")

with open(SAHI_CONFIG_PATH, 'r') as f:
    sahi_config = yaml.safe_load(f)

for preset in ['balanced', 'safety', 'max_recall', 'calib']:
    if preset in sahi_config['presets']:
        clean_rules = {}
        for k, v in sahi_config['presets'][preset]['class_rules'].items():
            clean_rules[str(k).strip()] = v
        sahi_config['presets'][preset]['class_rules'] = clean_rules
print("✅ Config loaded and cleaned.")

In [ ]:
def get_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

test_img_dir = Path(DATASET_YAML).parent / "test" / "images"
all_test_images = list(test_img_dir.glob("*.*"))
test_images = all_test_images[:MAX_IMAGES] if MAX_IMAGES else all_test_images
print(f"📸 Benchmarking on {len(test_images)} test images.")

In [ ]:
print("--- DIRECT INFERENCE BENCHMARK ---")
model = YOLO(MODEL_PATH)

# 1. mAP Evaluation
print("Running YOLO val for mAP (this may take a minute)...")
metrics = model.val(data=DATASET_YAML, split='test', imgsz=1024, batch=4, device=device, verbose=False)
print(f"🎯 Direct Test Mask mAP50: {metrics.seg.map50:.3f} | mAP50-95: {metrics.seg.map:.3f}")

# 2. Speed & Memory Loop
print(f"Running Speed/Memory loop on {len(test_images)} images...")
direct_times = []
mem_before = get_memory_mb()

for img_path in test_images:
    start = time.perf_counter()
    # Run inference
    _ = model(str(img_path), imgsz=1024, device=device, verbose=False)
    end = time.perf_counter()
    direct_times.append(end - start)

mem_after = get_memory_mb()
direct_mean_ms = np.mean(direct_times) * 1000
direct_p95_ms = np.percentile(direct_times, 95) * 1000
direct_mem_delta = mem_after - mem_before

print(f"⚡ Mean time: {direct_mean_ms:.1f} ms | p95: {direct_p95_ms:.1f} ms")
print(f"🧠 Memory delta: {direct_mem_delta:.1f} MB")

In [ ]:
print("\n--- SAHI INFERENCE BENCHMARK ---")
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=MODEL_PATH,
    confidence_threshold=0.01, # Keep low to capture all raw preds for preset filtering later
    device=device,
    image_size=1024
)

print(f"Running SAHI loop on {len(test_images)} images (this will be slower)...")
sahi_times = []
mem_before_sahi = get_memory_mb()
raw_sahi_results = []

for img_path in test_images:
    start = time.perf_counter()
    result = get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=1024,
        slice_width=1024,
        overlap_height_ratio=0.15,
        overlap_width_ratio=0.15,
        perform_standard_pred=False,
        postprocess_type="GREEDYNMM",
        postprocess_match_metric="IOS",
        postprocess_match_threshold=0.5,
        postprocess_class_agnostic=False,
        verbose=0,
    )
    end = time.perf_counter()
    
    sahi_times.append(end - start)
    raw_sahi_results.append((str(img_path), result.object_prediction_list))

mem_after_sahi = get_memory_mb()
sahi_mean_ms = np.mean(sahi_times) * 1000
sahi_p95_ms = np.percentile(sahi_times, 95) * 1000
sahi_mem_delta = mem_after_sahi - mem_before_sahi

print(f"⚡ Mean time: {sahi_mean_ms:.1f} ms | p95: {sahi_p95_ms:.1f} ms")
print(f"🧠 Memory delta: {sahi_mem_delta:.1f} MB")

In [ ]:
print("\n--- PRESET ANALYSIS (SAHI) ---")
presets_to_test = ['balanced', 'safety', 'max_recall']
summary_data = []

for preset_name in presets_to_test:
    rules = sahi_config['presets'][preset_name]['class_rules']
    counts = []
    for img_path, preds in raw_sahi_results:
        kept = 0
        for p in preds:
            class_id = str(p.category.id)
            if class_id in rules:
                rule = rules[class_id]
                if p.score.value >= rule['conf']:
                    # Use mask area if available, else bbox area as proxy
                    if hasattr(p, 'mask') and p.mask is not None and hasattr(p.mask, 'bool_mask'):
                        area = p.mask.bool_mask.sum()
                    else:
                        area = p.bbox.area
                    
                    if area >= rule['min_area']:
                        kept += 1
        counts.append(kept)
    
    avg_dets = np.mean(counts)
    print(f"Preset: {preset_name.upper():<12} | Avg detections per image: {avg_dets:.1f}")
    
    summary_data.append({
        "Mode": f"SAHI ({preset_name})",
        "Mean_ms": sahi_mean_ms,
        "p95_ms": sahi_p95_ms,
        "Mem_Delta_MB": sahi_mem_delta,
        "Avg_Detections": avg_dets
    })

# Add Direct to summary
summary_data.insert(0, {
    "Mode": "Direct (1024)",
    "Mean_ms": direct_mean_ms,
    "p95_ms": direct_p95_ms,
    "Mem_Delta_MB": direct_mem_delta,
    "Avg_Detections": "N/A (Use mAP)"
})

df = pd.DataFrame(summary_data)
print("\n📊 BENCHMARK SUMMARY")
try:
    # Try to use tabulate if installed for pretty printing
    print(df.to_markdown(index=False))
except ImportError:
    print(df.to_string(index=False))

In [ ]:
# 📓 Cell 4.5: Direct Inference Preset Analysis
print("\n--- PRESET ANALYSIS (DIRECT) ---")
direct_raw_results = []
for img_path in test_images:
    res = model(str(img_path), imgsz=1024, device=device, verbose=False)[0]
    direct_raw_results.append((str(img_path), res))

for preset_name in presets_to_test:
    rules = sahi_config['presets'][preset_name]['class_rules']
    counts = []
    for img_path, res in direct_raw_results:
        kept = 0
        if res.masks is not None and res.boxes is not None:
            masks_data = res.masks.data.cpu().numpy()
            boxes_data = res.boxes
            for i in range(len(boxes_data)):
                conf = float(boxes_data.conf[i])
                cls_id = str(int(boxes_data.cls[i]))
                # Calculate mask area in pixels
                mask_area = float(masks_data[i].sum()) 
                
                if cls_id in rules:
                    rule = rules[cls_id]
                    if conf >= rule['conf'] and mask_area >= rule['min_area']:
                        kept += 1
        counts.append(kept)
    
    avg_dets = np.mean(counts)
    print(f"Preset: {preset_name.upper():<12} | Direct Avg Detections: {avg_dets:.1f}")

In [ ]:
import pandas as pd

# Exact paths provided earlier for Models 1, 2, 3, and 4
models = {
    "Model 1 (Baseline)": "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/mlruns/1/63356a6890364538a0dad4c12bc2c43f/artifacts/results.csv",
    "Model 2 (Re-annotated)": "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/mlruns/1/dea31b142f8947d998d59cec41c980c3/artifacts/results.csv",
    "Model 3 (Control)": "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/mlruns/results.csv",
    "Model 4 (Diff LR)": "/Users/macbook/Documents/ITC8/Internship/AI Farm/Project/car_defect_detection/mlruns/1/15b2f231a76e4ae2a361e7d719e110e8/artifacts/results.csv"
}

for name, path in models.items():
    try:
        df = pd.read_csv(path)
        # Clean column names (remove leading/trailing spaces)
        df.columns = [c.strip() for c in df.columns]
        
        # Find the epoch with the best Mask mAP50
        best_idx = df['metrics/mAP50(M)'].idxmax()
        best_row = df.loc[best_idx]
        
        print(f"--- {name} (Best Val Epoch {int(best_row['epoch'])}) ---")
        print(f"Mask mAP50:    {best_row['metrics/mAP50(M)']:.3f}")
        print(f"Mask mAP50-95: {best_row['metrics/mAP50-95(M)']:.3f}")
        print(f"Precision:     {best_row['metrics/precision(M)']:.3f}")
        print(f"Recall:        {best_row['metrics/recall(M)']:.3f}\n")
    except Exception as e:
        print(f"Error reading {name}: {e}\n")